# Drivers 31 — Multivariate model
Gradient-boosted trees predicting WLR from the physical drivers
(rate-derived typology columns are **excluded** — target leakage), evaluated
with leave-one-region-out CV: the honest generalisation estimate under spatial
autocorrelation. Permutation importance is computed on held-out regions;
Moran's I quantifies how much spatial structure the drivers absorb.

In [1]:
import pandas as pd
from drivers.correlations import STATS_DIR

if not (STATS_DIR / "cv_scores.csv").exists():
    from drivers.multivariate import run
    run()
pd.read_csv(STATS_DIR / "cv_scores.csv")

,region,n_test,r2,mae
0,Northland,37267,-0.363290,1.105657
1,Canterbury,21875,-0.018160,0.453193
2,West Coast,20987,-0.184106,1.089615
3,Taranaki,17047,-0.194905,0.266954
4,Waikato,16587,-0.331647,0.598527
5,Otago,15424,-0.290710,0.616720
6,Bay of Plenty,13902,-0.624805,0.538868
7,Hawkes Bay,13494,-0.428757,0.481965
8,Southland,12380,-5.532069,1.222391
9,Auckland,11488,-1.098190,2.432171


## Headline result: no cross-region transfer, real local rank skill
Leave-one-region-out R² is negative in every region — driver relationships do
NOT generalise from 15 regions to the 16th. But with 10-km alongshore blocks
held out instead (local transfer), the model ranks eroding vs accreting
stretches well (Spearman ≈ 0.47) even though absolute variance explained stays
small. NZ coastal change is locally patterned, not nationally transferable.

In [2]:
b = STATS_DIR / "cv_scores_blocks.csv"
pd.read_csv(b) if b.exists() else "run the 10km-block CV (see qa/blockcv_run.log)"

,scheme,pooled_oof_r2,pooled_oof_spearman
0,10km_block_heldout,0.030446,0.472276


## What the model uses (held-out permutation importance)

In [3]:
pd.read_csv(STATS_DIR / "permutation_importance.csv", index_col=0).head(15)

,mean_importance,std_across_regions
beach_slope_face,0.068763,0.130643
backshore_mean,0.043906,0.077502
rain_mm_yr,0.042738,0.103085
hs_p99,0.035357,0.089712
tanbeta_nearshore,0.025744,0.107578
SHORE_TYPE,0.025600,0.173835
shaking_idx,0.020099,0.106685
HINTERLAND,0.019549,0.075386
dist_M65_km,0.018448,0.069211
landward_side,0.017241,0.036795


## Spatial structure absorbed

In [4]:
pd.read_csv(STATS_DIR / "morans_residuals.csv")

,variable,I,expected,p_perm,n
0,WLR,0.947492,-0.00002,0.01,50000
1,WLR_oof_residual,0.947606,-0.00002,0.01,50000


## Erosion/accretion sign classification (robustness)

In [5]:
pd.read_csv(STATS_DIR / "sign_classification.csv")

,region,balanced_acc
0,Northland,0.486936
1,Canterbury,0.705975
2,West Coast,0.622636
3,Taranaki,0.548552
4,Waikato,0.498339
5,Otago,0.597657
6,Bay of Plenty,0.450727
7,Hawkes Bay,0.609625
8,Southland,0.637886
9,Auckland,0.467343
